|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: break it on purpose<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'course/Part8_TheCapstone/7_incidents'))

import math, time
import torch
import lab
import copy, random
import cudalib
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, StaticCache

In the incident file you went from a symptom to a cause. Here you go the other
way. You put one fault into a working benchmark, step loop, pool, staging
buffer, kernel or budget, and you watch what it does.

The routine for each exercise is the same:

1. Read the fault.
2. **Write your prediction in the cell.** Answer the four questions.
3. Run the cell.
4. Write down where your prediction was wrong. This line is the one that
   teaches you.

The four questions:

- **Crash?** Does it raise an error, or does it run?
- **When?** Which step, which load, which context?
- **What?** What does the wrong result look like: a number above the roof, an
  idle GPU, a lost block, a wrong value?
- **Which guard?** Which check would catch it?

`lab.floor_ms` is the floor of one step from stage 28. The card numbers come
from `./vc info`: replace them with yours.

This notebook needs a GPU with about 9 GB free.

In [ ]:
### run this cell

MODEL = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).cuda().eval()
device = 'cuda'

BANDWIDTH, FLOPS = 294e9, 49e12          # ./vc info: the streaming bandwidth and the sustained bf16 rate
WEIGHTS = sum(p.numel() * p.element_size() for p in model.parameters())
PARAMS = sum(p.numel() for p in model.parameters())
KV = 2 * 28 * 8 * 128 * 2
def floor(computed, context):
  return lab.floor_ms(computed, context, WEIGHTS, PARAMS, KV, BANDWIDTH, FLOPS)
print(f'weights {WEIGHTS / 1e9:.2f} GB, one decode step at batch 1 >= {floor(1, 0):.1f} ms')

# Exercise 1: a floor that counts the prefix cache hits

16 requests share a system prompt of about 1,500 tokens, and each adds a
question of about 20 tokens. Prefill them two ways, and time each: with no
cache, and with the system prompt computed once and reused (a prefix cache).
Then compute the ratio floor / time two ways: with the floor of every prompt
token of every request, as the benchmark of Ticket 1 does, and with the floor
of the tokens that the engine really computed.

Predict the four ratios.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
SYSTEM = ('You are the support assistant of a bank. Answer politely, in short paragraphs, and never share '
          'account data. ' * 60)
questions = [f'Question {i}: how do I change the address on my account number {1000 + i}?' for i in range(16)]
system_ids = tokenizer(SYSTEM, return_tensors='pt').input_ids.to(device)
question_ids = [tokenizer(q, return_tensors='pt').input_ids.to(device) for q in questions]
S = system_ids.shape[1]
print(f'system prompt {S} tokens, questions {[q.shape[1] for q in question_ids][:4]}... tokens')

def run(prefix_cache):
  with torch.inference_mode():
    shared = DynamicCache()
    if prefix_cache:
      model(system_ids, past_key_values=shared, use_cache=True, logits_to_keep=1)
    for q in question_ids:
      if prefix_cache:
        cache = copy.deepcopy(shared)
        model(q, past_key_values=cache, use_cache=True, logits_to_keep=1)
      else:
        model(torch.cat([system_ids, q], dim=1), use_cache=True, logits_to_keep=1)

run(True)                                                              # warm up
for prefix_cache in (False, True):
  torch.cuda.synchronize()
  start = time.perf_counter()
  run(prefix_cache)
  torch.cuda.synchronize()
  ms = (time.perf_counter() - start) * 1000
  every_token = sum(floor(S + q.shape[1], 0) for q in question_ids)                  # THE FAULT
  computed = ((floor(S, 0) if prefix_cache else 0)
              + sum(floor(q.shape[1], S) if prefix_cache else floor(S + q.shape[1], 0) for q in question_ids))
  print(f'prefix cache {str(prefix_cache):5s}: {ms:6.0f} ms   floor of every prompt token / time = '
        f'{every_token / ms:5.0%}   floor of the computed tokens / time = {computed / ms:5.0%}')

# Exercise 2: the bytes of a long-context step

Batch 8, and a static cache that holds 256 tokens, then 4,096 tokens of
context for each sequence. Time one decode step at each context. The step at
256 is almost all weights. The difference is the KV cache. Then predict what
int8 weights (half the weight bytes) and an FP8 KV cache (half the KV bytes)
would each give at 4,096.

This is Ticket 2. Predict both speedups at 4,096.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
@torch.inference_mode()
def step_ms(batch, context):
  cache = StaticCache(config=model.config, max_cache_len=context + 64)
  model(torch.randint(0, 1000, (batch, context), device=device), past_key_values=cache,
        use_cache=True, logits_to_keep=1)
  token = torch.randint(0, 1000, (batch, 1), device=device)
  ms = lab.ms_per_call(lambda: model(token, past_key_values=cache, use_cache=True), iters=20)
  del cache
  torch.cuda.empty_cache()
  return ms

short, long = step_ms(8, 256), step_ms(8, 4096)
weight_part, kv_part = short, long - short
print(f'batch 8: {short:.1f} ms at 256 tokens, {long:.1f} ms at 4,096 tokens')
print(f'at 4,096: the weights about {weight_part:.1f} ms ({weight_part / long:.0%}), the KV about {kv_part:.1f} ms ({kv_part / long:.0%})')
print(f'int8 weights -> {long / (weight_part / 2 + kv_part):.2f}x    FP8 KV cache -> '
      f'{long / (weight_part + kv_part / 2):.2f}x    both -> {long / (weight_part / 2 + kv_part / 2):.2f}x')

# Exercise 3: CPU work between the steps

A greedy loop at batch 1, 60 tokens. After each step the engine does 4 ms of
CPU work: the scheduler, the detokenizer. Time three loops: no CPU work, the
CPU work after each step (in series, as in Ticket 3), and the CPU work for
token n while the GPU computes token n + 1.

Predict the ms of each step.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
def cpu_work(ms=4.0):                                                  # busy, like Python work
  end = time.perf_counter() + ms / 1000
  while time.perf_counter() < end:
    pass

@torch.inference_mode()
def loop(mode, tokens=60):
  ids = tokenizer('The three largest cities in Japan are', return_tensors='pt').input_ids.to(device)
  out = model(ids, use_cache=True)
  cache, token = out.past_key_values, out.logits[:, -1:].argmax(-1)
  host = torch.empty(1, 1, dtype=torch.long, pin_memory=True)
  torch.cuda.synchronize()
  start = time.perf_counter()
  for _ in range(tokens):
    out = model(token, past_key_values=cache, use_cache=True)          # the GPU starts at once
    token = out.logits[:, -1:].argmax(-1)
    if mode == 'series':
      int(token)                                                       # wait for the GPU ...
      cpu_work()                                                       # THE FAULT: ... then work
    elif mode == 'overlap':
      cpu_work()                                                       # work while the GPU runs
      host.copy_(token, non_blocking=True)
  torch.cuda.synchronize()
  return (time.perf_counter() - start) / tokens * 1000

for mode in ('none', 'series', 'overlap'):
  print(f'CPU work {mode:8s}: {loop(mode):5.1f} ms per step')

# Exercise 4: preemption keeps the last block

A pool of 8,000 blocks. 300 sequences grow to random lengths, and 1,250 times
one of them is preempted and later resumed. The preemption frees every block
except the last one, as the code of Ticket 4 does. At the end every sequence
finishes.

Predict the free blocks at the end.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
pool = lab.Pool(8000)
rng = random.Random(0)
lengths = {seq: rng.randint(20, 300) for seq in range(300)}
for seq, n in lengths.items():
  pool.grow(seq, n)

def preempt(seq):
  table = pool.tables.pop(seq)
  pool.free_list += table[:-1]                                         # THE FAULT: the last block stays

for _ in range(1250):
  seq = rng.randrange(300)
  preempt(seq)
  pool.grow(seq, lengths[seq])                                         # resume: allocate again
for seq in list(pool.tables):
  pool.release(seq)
print(f'free at the end: {pool.num_free()} of {pool.total}; lost: {pool.total - pool.num_free()}')

# Exercise 5: one pinned staging buffer, reused at once

The engine copies the block table of each step from a pinned CPU buffer to
the GPU with `non_blocking=True`, and then at once writes the table of the
next step into the same buffer. Make the GPU busy first, as a heavy step
does. Then check what the GPU received. Then do the same with an event that
the CPU waits on before it reuses the buffer.

This is Ticket 5. Predict what the GPU received in each case.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
busy = torch.randn(4096, 4096, device=device)
host = torch.empty(8, dtype=torch.long, pin_memory=True)
table = torch.empty(8, dtype=torch.long, device=device)
done = torch.cuda.Event()

for guard in (False, True):
  for _ in range(20):
    busy @ busy                                                        # the GPU is behind the CPU
  host[:] = torch.arange(8)                                            # the table of step n
  table.copy_(host, non_blocking=True)
  done.record()
  if guard:
    done.synchronize()                                                 # wait until the copy has read the buffer
  host[:] = 100 + torch.arange(8)                                      # THE FAULT: step n + 1 writes at once
  torch.cuda.synchronize()
  print(f'{"with the event" if guard else "no guard      "}: the GPU received {table.tolist()} (step n sent 0 to 7)')

# Exercise 6: a shared tile of 128 floats per row

A kernel keeps a tile of 32 x 128 floats in shared memory, and each thread of
a warp reads one **column**, many times. Then the same kernel with rows of 129
floats. The first run compiles the kernel.

This is Ticket 6. Predict the ratio of the two times.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
KERNEL = r"""
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h>

template <int WIDTH>
__global__ void column_reads(float* out, int repeats) {
  __shared__ float tile[32][WIDTH];
  int lane = threadIdx.x;
  for (int c = 0; c < 128; ++c) tile[lane][c] = lane * 0.5f + c;
  __syncwarp();
  float sum = 0.f;
  for (int r = 0; r < repeats; ++r)
    for (int c = 0; c < 128; ++c) sum += tile[lane][(c + r) & 127];     // 32 lanes, 32 rows, one column
  out[blockIdx.x * 32 + lane] = sum;
}

void run(torch::Tensor out, int64_t width, int64_t repeats) {
  int blocks = out.numel() / 32;
  auto stream = at::cuda::getCurrentCUDAStream();
  if (width == 128) column_reads<128><<<blocks, 32, 0, stream>>>(out.data_ptr<float>(), repeats);
  else column_reads<129><<<blocks, 32, 0, stream>>>(out.data_ptr<float>(), repeats);
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("run", &run); }
"""
kernel = cudalib.build_source('inc8_banks', KERNEL)
out = torch.empty(4096 * 32, device=device)
times = {}
for width in (128, 129):                                               # 128: THE FAULT
  times[width] = lab.ms_per_call(lambda: kernel.run(out, width, 64), iters=10)
  print(f'rows of {width} floats: {times[width]:6.2f} ms, the bank of column 0 for the 32 lanes: '
        f'{len({(lane * width) % 32 for lane in range(32)})} different bank(s)')
print(f'ratio: {times[128] / times[129]:.1f}x')

# Exercise 7: more running sequences than the token budget

A simulated scheduler: `max_num_seqs` is 256, the token budget is 192, and
decodes go first. At the peak 230 sequences are running, and a new request
arrives every step. Run 200 steps, then do the same with a budget of 768.

This is Ticket 7. Predict the tokens that each running sequence gets, and
what happens to the new requests.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
def simulate(budget, max_seqs=256, steps=200, seed=0):
  rng = random.Random(seed)
  running = [rng.randint(1, 400) for _ in range(230)]                   # the tokens that each still decodes
  waiting, served, got, slots = [], 0, 0, 0
  for step in range(steps):
    waiting.append(300)                                                 # a new prompt of 300 tokens
    rng.shuffle(running)
    decodes = min(len(running), budget)                                 # decodes first
    got, slots = got + decodes, slots + len(running)
    running = [left - 1 if i < decodes else left for i, left in enumerate(running)]
    running = [left for left in running if left > 0]
    room = budget - decodes                                             # then the prefills
    while waiting and room > 0 and len(running) < max_seqs:
      take = min(waiting[0], room)
      waiting[0] -= take
      room -= take
      if waiting[0] == 0:
        waiting.pop(0)
        running.append(200)
        served += 1
  return got / slots, served, len(waiting)

for budget in (192, 768):                                              # 192: THE FAULT
  rate, served, waiting = simulate(budget)
  print(f'budget {budget}: a running sequence got a token in {rate:.0%} of the steps; '
        f'{served} new requests got a first token, {waiting} still wait')

# Exercise 8: three mystery benchmarks

The module `mystery.py` holds three functions: `ratio_a`, `ratio_b` and
`ratio_c`. Each takes a step log and the card, and returns the ratio floor /
time of stage 28. A step in the log is a dict: `computed` (tokens run through
the model), `cached` (prompt tokens that the prefix cache served), `context`
(tokens of KV that the step read) and `ms`. Each one has one fault. **Do not
open the file.**

`floor(computed, context)` from the load cell is the correct floor of one
step, so you can compute the true ratio of any log you build.

The cell below runs them on a simple log: identical decode steps, short
context, no prefix cache. All three agree within 1%.

For each function:

1. Build a step log that reaches the fault, and compute the true ratio.
2. Write your diagnosis: the fault, and the log that proved it.
3. Only then, open `mystery.py` and check.

A hint about the method: one fault needs the prefix cache. One fault needs a
long context. One fault needs steps of very different sizes.

In [ ]:
from mystery import ratio_a, ratio_b, ratio_c

card = dict(weight_bytes=WEIGHTS, params=PARAMS, kv_per_token=KV, bandwidth=BANDWIDTH, flops=FLOPS)
simple = [dict(computed=8, cached=0, context=8 * 16, ms=13.0) for _ in range(100)]
true = sum(floor(s['computed'], s['context']) for s in simple) / sum(s['ms'] for s in simple)
print(f'true {true:.3f}   a {ratio_a(simple, **card):.3f}   b {ratio_b(simple, **card):.3f}   c {ratio_c(simple, **card):.3f}')

**Your diagnosis**

- `ratio_a`: the fault, and the experiment that proves it:
- `ratio_b`: the fault, and the experiment that proves it:
- `ratio_c`: the fault, and the experiment that proves it:

# Your fingerprint table

Fill in this table from what you saw, not from what you predicted.

| Fault | Crash? | When it shows | What it looks like | The guard |
|---|---|---|---|---|
| a floor that counts the prefix cache hits | | | | |
| int8 weights at long context | | | | |
| CPU work in series with the GPU | | | | |
| preemption keeps the last block | | | | |
| one pinned staging buffer, reused at once | | | | |
| a stride of 128 floats in shared memory | | | | |
| more running sequences than the budget | | | | |